In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1MS1fS1zIl7QPQYq9nYhJZ4JcUdbhwjit", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))


# Interleaved 1F1B: Virtual Pipeline Stages -- Vizuara

## What You Will Learn

In this notebook, you will:

1. **Understand virtual pipeline stages** and non-contiguous layer assignment
2. **Implement the interleaved 1F1B schedule** from scratch
3. **Verify the bubble reduction** by a factor of $v$
4. **Analyze the communication overhead** trade-off
5. **Compare all four schedules** (Naive, GPipe, 1F1B, Interleaved 1F1B) head to head

**Estimated time**: 40-60 minutes

**Prerequisites**: Notebooks 01 and 02 (pipeline bubble, GPipe, 1F1B)

In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

%matplotlib inline
np.random.seed(42)

print("Setup complete!")
print("In this notebook we explore Interleaved 1F1B (virtual pipeline stages).")

In [ ]:
# Core data structures (from previous notebooks)

@dataclass
class PipelineStage:
    stage_id: int
    forward_time: float
    backward_time: float

@dataclass
class ScheduleEvent:
    stage_id: int
    microbatch_id: int
    is_forward: bool
    start_time: float
    end_time: float
    virtual_stage: int = 0  # NEW: which virtual stage this belongs to

def create_uniform_pipeline(num_stages, forward_time=100.0):
    return [
        PipelineStage(stage_id=i, forward_time=forward_time, backward_time=forward_time * 2.0)
        for i in range(num_stages)
    ]

def compute_bubble_fraction(events, num_stages):
    total_time = max(e.end_time for e in events)
    total_gpu_time = num_stages * total_time
    busy = sum(e.end_time - e.start_time for e in events)
    idle = total_gpu_time - busy
    return {
        "total_time": total_time,
        "bubble_fraction": idle / total_gpu_time,
        "total_busy": busy,
        "total_idle": idle
    }

print("Core data structures loaded.")

In [ ]:
#@title 🎧 Listen: Virtual Stages Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_virtual_stages_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. Understanding Virtual Pipeline Stages

In standard pipeline parallelism, each GPU handles one contiguous block of layers:
- GPU 0: Layers 1-8
- GPU 1: Layers 9-16
- GPU 2: Layers 17-24
- GPU 3: Layers 25-32

In **interleaved** pipeline parallelism, each GPU handles $v$ **non-contiguous** blocks. With $v = 2$:
- GPU 0: Layers 1-4 **and** Layers 17-20
- GPU 1: Layers 5-8 **and** Layers 21-24
- GPU 2: Layers 9-12 **and** Layers 25-28
- GPU 3: Layers 13-16 **and** Layers 29-32

The pipeline now has $P \times v = 8$ **virtual stages**, and each virtual stage processes only $L / (P \cdot v)$ layers.

In [ ]:
#@title 🎧 What to Look For: Visualize Layers
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_visualize_layers.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def visualize_layer_assignment(L: int, P: int, v: int):
    """Visualize how layers are assigned to GPUs with virtual stages."""
    layers_per_virtual_stage = L // (P * v)
    total_virtual_stages = P * v

    print(f"Model: {L} layers, {P} GPUs, {v} virtual stages per GPU")
    print(f"Total virtual stages: {total_virtual_stages}")
    print(f"Layers per virtual stage: {layers_per_virtual_stage}")
    print()

    # Assign layers to virtual stages
    assignments = {}  # gpu_id -> list of (virtual_stage_id, layer_start, layer_end)
    for gpu in range(P):
        assignments[gpu] = []

    for vs in range(total_virtual_stages):
        gpu = vs % P
        layer_start = vs * layers_per_virtual_stage + 1
        layer_end = layer_start + layers_per_virtual_stage - 1
        assignments[gpu].append((vs, layer_start, layer_end))

    # Display
    colors = plt.cm.Set3(np.linspace(0, 1, total_virtual_stages))

    fig, ax = plt.subplots(figsize=(14, 3))

    for gpu in range(P):
        for vs, layer_start, layer_end in assignments[gpu]:
            # Draw each layer block
            x = layer_start - 1
            width = layer_end - layer_start + 1
            rect = mpatches.FancyBboxPatch((x, P - 1 - gpu - 0.4), width, 0.8,
                                           boxstyle="round,pad=0.05",
                                           facecolor=colors[vs], edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            ax.text(x + width / 2, P - 1 - gpu,
                    f"VS{vs}\nL{layer_start}-{layer_end}",
                    ha='center', va='center', fontsize=8, fontweight='bold')

    ax.set_xlim(-0.5, L + 0.5)
    ax.set_ylim(-0.6, P - 0.4)
    ax.set_yticks(range(P))
    ax.set_yticklabels([f'GPU {P-1-i}' for i in range(P)])
    ax.set_xlabel('Layer Index', fontsize=11)
    ax.set_title(f'Layer Assignment: {P} GPUs, v={v} virtual stages per GPU',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Print the data flow path
    print("Data flow path:")
    path = []
    for vs in range(total_virtual_stages):
        gpu = vs % P
        path.append(f"GPU{gpu}(VS{vs})")
    print(" -> ".join(path))

    return assignments

# Standard (v=1)
print("=" * 60)
print("STANDARD PIPELINE (v=1)")
print("=" * 60)
visualize_layer_assignment(32, 4, 1)

print()

# Interleaved (v=2)
print("=" * 60)
print("INTERLEAVED PIPELINE (v=2)")
print("=" * 60)
visualize_layer_assignment(32, 4, 2)

In [ ]:
#@title 🎧 Listen: Why Interleaving Helps
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_why_interleaving_helps.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Why Does Interleaving Help?

The bubble fraction with standard 1F1B is:
$$\text{Bubble} = \frac{P - 1}{P - 1 + M}$$

With $v$ virtual stages per GPU, each virtual stage is $1/v$ the size of a full stage, so each forward/backward pass takes $1/v$ the time. The bubble fraction becomes:

$$\text{Bubble (interleaved)} = \frac{P - 1}{v \cdot (P - 1 + M)}$$

This is a factor of $v$ smaller! Let us verify this.

In [ ]:
#@title 🎧 Code Walkthrough: Theoretical Bubble
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_theoretical_bubble.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Theoretical bubble fractions
P = 4
M = 12

print(f"Bubble fractions for P={P}, M={M}:")
print(f"{'v':>4} {'Formula':>40} {'Bubble':>10}")
print("-" * 58)

for v in [1, 2, 4, 8]:
    bubble = (P - 1) / (v * (P - 1 + M))
    formula = f"(P-1) / (v*(P-1+M)) = {P-1} / ({v}*{P-1+M})"
    print(f"{v:>4} {formula:>40} {bubble:>9.1%}")

print("\nEach doubling of v halves the bubble!")

In [ ]:
#@title 🎧 Listen: Implementing Interleaved
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_implementing_interleaved.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Implementing the Interleaved 1F1B Schedule

The interleaved schedule is more complex because:
1. Each physical GPU runs multiple virtual stages.
2. Data flows through $P \times v$ virtual stages total.
3. Each virtual stage's computation is $1/v$ of a full stage.

In [ ]:
#@title 🎧 Code Walkthrough: Schedule Interleaved Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_schedule_interleaved_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def schedule_interleaved_1f1b(num_physical_gpus: int, num_virtual_per_gpu: int,
                               num_microbatches: int,
                               forward_time_per_full_stage: float = 100.0
                               ) -> List[ScheduleEvent]:
    """Interleaved 1F1B with virtual pipeline stages.

    Each GPU handles v virtual stages. Each virtual stage takes 1/v the time.
    The total pipeline has P*v virtual stages.
    """
    P = num_physical_gpus
    v = num_virtual_per_gpu
    M = num_microbatches
    total_vs = P * v  # total virtual stages

    # Time per virtual stage (1/v of full stage)
    fwd_time = forward_time_per_full_stage / v
    bwd_time = fwd_time * 2.0

    events = []

    # Map virtual stage -> physical GPU
    vs_to_gpu = {vs: vs % P for vs in range(total_vs)}

    # Track when each physical GPU is free
    gpu_free_at = [0.0] * P

    # Track forward/backward completion for dependency
    fwd_done = {}  # (virtual_stage, mb) -> end_time
    bwd_done = {}  # (virtual_stage, mb) -> end_time

    # 1F1B schedule across virtual stages
    # Warmup: fill pipeline across all virtual stages
    # Then steady state: 1F + 1B per physical GPU time slot

    # Simple approach: schedule virtual stages in order
    # Each micro-batch flows through VS 0, VS 1, ..., VS (P*v - 1)

    # Phase 1: All forward passes (pipelined)
    for mb in range(M):
        for vs in range(total_vs):
            gpu = vs_to_gpu[vs]
            earliest = gpu_free_at[gpu]

            # Must wait for previous virtual stage to finish this micro-batch
            if vs > 0:
                earliest = max(earliest, fwd_done[(vs - 1, mb)])

            start = earliest
            end = start + fwd_time

            events.append(ScheduleEvent(
                stage_id=gpu, microbatch_id=mb,
                is_forward=True, start_time=start, end_time=end,
                virtual_stage=vs
            ))
            gpu_free_at[gpu] = end
            fwd_done[(vs, mb)] = end

    # Phase 2: All backward passes (reverse order)
    for mb in range(M - 1, -1, -1):
        for vs in range(total_vs - 1, -1, -1):
            gpu = vs_to_gpu[vs]
            earliest = gpu_free_at[gpu]

            if vs < total_vs - 1:
                earliest = max(earliest, bwd_done[(vs + 1, mb)])

            start = earliest
            end = start + bwd_time

            events.append(ScheduleEvent(
                stage_id=gpu, microbatch_id=mb,
                is_forward=False, start_time=start, end_time=end,
                virtual_stage=vs
            ))
            gpu_free_at[gpu] = end
            bwd_done[(vs, mb)] = end

    return events

# Test with P=4, v=2, M=8
P, v, M = 4, 2, 8
events_interleaved = schedule_interleaved_1f1b(P, v, M, forward_time_per_full_stage=100.0)
stats_interleaved = compute_bubble_fraction(events_interleaved, P)

print(f"Interleaved 1F1B (P={P}, v={v}, M={M}):")
print(f"  Total virtual stages: {P * v}")
print(f"  Total time: {stats_interleaved['total_time']:.0f} ms")
print(f"  Bubble fraction: {stats_interleaved['bubble_fraction']:.1%}")

# Compare with standard 1F1B theory
standard_bubble = (P - 1) / (P - 1 + M)
interleaved_theory = (P - 1) / (v * (P - 1 + M))
print(f"\n  Standard 1F1B theory: {standard_bubble:.1%}")
print(f"  Interleaved theory:   {interleaved_theory:.1%}")
print(f"  Reduction factor:     {standard_bubble / interleaved_theory:.1f}x")

In [ ]:
#@title 🎧 Transition: Visualizing Interleaved Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_visualizing_interleaved_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Visualizing the Interleaved Schedule

Let us see how the interleaved schedule looks. We will color-code by virtual stage to see how each GPU interleaves work from different parts of the model.

In [ ]:
#@title 🎧 What to Look For: Plot Interleaved Timeline
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_plot_interleaved_timeline.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def plot_interleaved_timeline(events: List[ScheduleEvent], num_gpus: int,
                               num_virtual: int, title: str, figsize=(20, 5)):
    """Visualize interleaved pipeline with virtual stage colors."""
    fig, ax = plt.subplots(figsize=figsize)
    total_time = max(e.end_time for e in events)

    # Color by virtual stage
    total_vs = num_gpus * num_virtual
    fwd_colors = plt.cm.Blues(np.linspace(0.3, 0.9, total_vs))
    bwd_colors = plt.cm.Reds(np.linspace(0.3, 0.9, total_vs))

    # Draw idle background
    for s in range(num_gpus):
        ax.barh(s, total_time, left=0, height=0.7, color='#E8E8E8',
                edgecolor='white', linewidth=0.5)

    # Draw events
    for e in events:
        vs = e.virtual_stage
        color = fwd_colors[vs] if e.is_forward else bwd_colors[vs]
        duration = e.end_time - e.start_time
        ax.barh(e.stage_id, duration, left=e.start_time, height=0.7,
                color=color, edgecolor='white', linewidth=0.5)

        if duration > 20:
            label = f"{'F' if e.is_forward else 'B'}{e.microbatch_id}"
            ax.text(e.start_time + duration / 2, e.stage_id, label,
                    ha='center', va='center', fontsize=6, fontweight='bold', color='white')

    ax.set_yticks(range(num_gpus))
    ax.set_yticklabels([f'GPU {i}' for i in range(num_gpus)])
    ax.set_xlabel('Time (ms)', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.invert_yaxis()

    stats = compute_bubble_fraction(events, num_gpus)
    ax.text(0.02, 0.02, f"Bubble: {stats['bubble_fraction']:.1%}",
            transform=ax.transAxes, fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()

# Visualize
plot_interleaved_timeline(events_interleaved, P, v,
                          f"Interleaved 1F1B (P={P}, v={v}, M={M})")

In [ ]:
#@title 🎧 Code Walkthrough: Standard Comparison
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_standard_comparison.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# For comparison, let us also schedule and visualize standard 1F1B with same P, M

def schedule_1f1b(stages, num_microbatches):
    """Standard 1F1B from Notebook 02."""
    events = []
    P = len(stages)
    M = num_microbatches
    stage_free_at = [0.0] * P
    fwd_done = {}
    bwd_done = {}

    for s in range(P):
        num_warmup = P - 1 - s
        num_steady = M - num_warmup
        fwd_idx = 0
        bwd_idx = 0

        for i in range(num_warmup):
            mb = fwd_idx
            earliest = stage_free_at[s]
            if s > 0:
                earliest = max(earliest, fwd_done[(s - 1, mb)])
            start = earliest
            end = start + stages[s].forward_time
            events.append(ScheduleEvent(stage_id=s, microbatch_id=mb,
                                        is_forward=True, start_time=start, end_time=end))
            stage_free_at[s] = end
            fwd_done[(s, mb)] = end
            fwd_idx += 1

        for i in range(num_steady):
            mb_fwd = fwd_idx
            earliest_fwd = stage_free_at[s]
            if s > 0:
                earliest_fwd = max(earliest_fwd, fwd_done[(s - 1, mb_fwd)])
            start_fwd = earliest_fwd
            end_fwd = start_fwd + stages[s].forward_time
            events.append(ScheduleEvent(stage_id=s, microbatch_id=mb_fwd,
                                        is_forward=True, start_time=start_fwd, end_time=end_fwd))
            stage_free_at[s] = end_fwd
            fwd_done[(s, mb_fwd)] = end_fwd
            fwd_idx += 1

            mb_bwd = bwd_idx
            earliest_bwd = stage_free_at[s]
            if s < P - 1:
                earliest_bwd = max(earliest_bwd, bwd_done[(s + 1, mb_bwd)])
            start_bwd = earliest_bwd
            end_bwd = start_bwd + stages[s].backward_time
            events.append(ScheduleEvent(stage_id=s, microbatch_id=mb_bwd,
                                        is_forward=False, start_time=start_bwd, end_time=end_bwd))
            stage_free_at[s] = end_bwd
            bwd_done[(s, mb_bwd)] = end_bwd
            bwd_idx += 1

        for i in range(num_warmup):
            mb = bwd_idx
            earliest = stage_free_at[s]
            if s < P - 1:
                earliest = max(earliest, bwd_done[(s + 1, mb)])
            start = earliest
            end = start + stages[s].backward_time
            events.append(ScheduleEvent(stage_id=s, microbatch_id=mb,
                                        is_forward=False, start_time=start, end_time=end))
            stage_free_at[s] = end
            bwd_done[(s, mb)] = end
            bwd_idx += 1

    return events

# Standard 1F1B for comparison
stages_standard = create_uniform_pipeline(P, forward_time=100.0)
events_standard = schedule_1f1b(stages_standard, M)
stats_standard = compute_bubble_fraction(events_standard, P)

print(f"Standard 1F1B (P={P}, M={M}):  Bubble = {stats_standard['bubble_fraction']:.1%}")
print(f"Interleaved   (P={P}, v={v}, M={M}): Bubble = {stats_interleaved['bubble_fraction']:.1%}")

In [ ]:
#@title 🎧 Listen: Comm Tradeoff Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_comm_tradeoff_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. The Communication Trade-Off

Interleaved 1F1B reduces the bubble, but at a cost: more point-to-point communications.

With standard 1F1B, each micro-batch passes through $P$ stages (one activation transfer between each pair). With $v$ virtual stages per GPU, each micro-batch passes through $P \times v$ virtual stages, requiring $v$ times as many transfers.

However, each transfer is **smaller** (since each virtual stage processes fewer layers). The key question is whether the communication overhead outweighs the bubble reduction.

In [ ]:
#@title 🎧 Code Walkthrough: Comm Analysis Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_comm_analysis_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def analyze_communication(P: int, v: int, M: int,
                           hidden_dim: int = 4096,
                           seq_len: int = 4096,
                           batch_size: int = 1,
                           dtype_bytes: int = 2,
                           bandwidth_gbps: float = 300.0) -> dict:
    """Analyze communication overhead for interleaved pipeline.

    Args:
        bandwidth_gbps: Link bandwidth in GB/s (300 GB/s for NVLink, ~25 GB/s for IB)
    """
    # Activation size per transfer (same regardless of v)
    act_bytes = batch_size * seq_len * hidden_dim * dtype_bytes
    act_mb = act_bytes / (1024 ** 2)

    # Number of point-to-point transfers per micro-batch
    transfers_standard = P - 1  # one per stage boundary
    transfers_interleaved = P * v - 1  # one per virtual stage boundary

    # Total transfers for all micro-batches (forward + backward)
    total_standard = 2 * M * transfers_standard  # 2x for fwd + bwd
    total_interleaved = 2 * M * transfers_interleaved

    # Communication time per transfer
    comm_time_ms = act_mb / (bandwidth_gbps * 1024 / 1000)  # convert GB/s to MB/ms

    return {
        "act_size_mb": act_mb,
        "transfers_per_mb_standard": transfers_standard,
        "transfers_per_mb_interleaved": transfers_interleaved,
        "total_transfers_standard": total_standard,
        "total_transfers_interleaved": total_interleaved,
        "comm_time_per_transfer_ms": comm_time_ms,
        "total_comm_standard_ms": total_standard * comm_time_ms,
        "total_comm_interleaved_ms": total_interleaved * comm_time_ms,
        "comm_increase_factor": total_interleaved / total_standard
    }

# Analyze for different v values
P, M = 4, 16

print(f"Communication Analysis (P={P}, M={M}, LLaMA-7B config)\n")
print(f"{'v':>4} {'Transfers/MB':>14} {'Total Transfers':>16} {'Comm Time':>12} {'Bubble':>10}")
print("-" * 62)

for v in [1, 2, 4]:
    comm = analyze_communication(P, v, M, bandwidth_gbps=300.0)  # NVLink
    bubble = (P - 1) / (v * (P - 1 + M)) * 100

    if v == 1:
        transfers = comm['transfers_per_mb_standard']
        total = comm['total_transfers_standard']
        time_ms = comm['total_comm_standard_ms']
    else:
        transfers = comm['transfers_per_mb_interleaved']
        total = comm['total_transfers_interleaved']
        time_ms = comm['total_comm_interleaved_ms']

    print(f"{v:>4} {transfers:>14} {total:>16} {time_ms:>10.1f}ms {bubble:>9.1f}%")

print(f"\nWith NVLink (300 GB/s), communication is cheap.")
print(f"The bubble reduction from v=2 or v=4 is almost always worth it intra-node.")

In [ ]:
# Now compare with slow interconnect (cross-node InfiniBand)
print(f"Communication Analysis (P={P}, M={M}, cross-node InfiniBand)\n")
print(f"{'v':>4} {'Comm Time (NVLink)':>20} {'Comm Time (IB)':>20} {'Bubble':>10}")
print("-" * 56)

for v in [1, 2, 4]:
    comm_nvlink = analyze_communication(P, v, M, bandwidth_gbps=300.0)
    comm_ib = analyze_communication(P, v, M, bandwidth_gbps=25.0)
    bubble = (P - 1) / (v * (P - 1 + M)) * 100

    if v == 1:
        time_nvlink = comm_nvlink['total_comm_standard_ms']
        time_ib = comm_ib['total_comm_standard_ms']
    else:
        time_nvlink = comm_nvlink['total_comm_interleaved_ms']
        time_ib = comm_ib['total_comm_interleaved_ms']

    print(f"{v:>4} {time_nvlink:>18.1f}ms {time_ib:>18.1f}ms {bubble:>9.1f}%")

print(f"\nWith InfiniBand (~25 GB/s), communication is 12x more expensive.")
print(f"At v=4 with IB, the extra communication may negate the bubble savings.")
print(f"This is why interleaved 1F1B is best suited for INTRA-NODE parallelism.")

In [ ]:
#@title 🎧 Transition: Grand Comparison Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_grand_comparison_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Grand Comparison: All Four Schedules

Let us put all four pipeline schedules side by side and compare their bubble fractions and characteristics.

In [ ]:
#@title 🎧 What to Look For: Comparison Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_comparison_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
P = 4

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Bubble vs M for different schedules
M_range = np.arange(1, 65)
ax1 = axes[0]

# Naive
naive_bubble = np.full_like(M_range, (P - 1) / P, dtype=float) * 100
ax1.plot(M_range, naive_bubble, '--', color='gray', linewidth=2, label='Naive')

# GPipe / 1F1B (same bubble)
gpipe_bubble = (P - 1) / (P - 1 + M_range) * 100
ax1.plot(M_range, gpipe_bubble, '-', color='#E8634F', linewidth=2, label='GPipe / 1F1B (v=1)')

# Interleaved v=2
interleaved_v2 = (P - 1) / (2 * (P - 1 + M_range)) * 100
ax1.plot(M_range, interleaved_v2, '-', color='#4A90D9', linewidth=2, label='Interleaved (v=2)')

# Interleaved v=4
interleaved_v4 = (P - 1) / (4 * (P - 1 + M_range)) * 100
ax1.plot(M_range, interleaved_v4, '-', color='#2ECC71', linewidth=2, label='Interleaved (v=4)')

ax1.axhline(y=10, color='gray', linestyle=':', alpha=0.5)
ax1.text(55, 11, '10%', fontsize=9, color='gray')
ax1.set_xlabel('Number of Micro-Batches (M)', fontsize=11)
ax1.set_ylabel('Bubble Fraction (%)', fontsize=11)
ax1.set_title('Bubble Fraction Comparison (P=4)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 80)

# Right: Summary table as a bar chart for M=16
ax2 = axes[1]
M_fixed = 16

schedules = ['Naive', 'GPipe', '1F1B', 'Interleaved\n(v=2)', 'Interleaved\n(v=4)']
bubbles = [
    (P - 1) / P * 100,
    (P - 1) / (P - 1 + M_fixed) * 100,
    (P - 1) / (P - 1 + M_fixed) * 100,
    (P - 1) / (2 * (P - 1 + M_fixed)) * 100,
    (P - 1) / (4 * (P - 1 + M_fixed)) * 100,
]
peak_memory = [
    1,  # naive: 1 MB
    M_fixed,
    P,
    P,
    P,
]

colors = ['gray', '#E8634F', '#4A90D9', '#2ECC71', '#9B59B6']
bars = ax2.bar(schedules, bubbles, color=colors, edgecolor='white', linewidth=1.5)

# Add value labels
for bar, b, mem in zip(bars, bubbles, peak_memory):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{b:.1f}%\nmem={mem}', ha='center', fontsize=9, fontweight='bold')

ax2.set_ylabel('Bubble Fraction (%)', fontsize=11)
ax2.set_title(f'All Schedules (P={P}, M={M_fixed})', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Code Walkthrough: Comparison Table
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_comparison_table.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Print the complete summary table
P = 4
M = 16

print(f"Complete Schedule Comparison (P={P}, M={M})")
print(f"{'='*80}")
print(f"{'Schedule':<25} {'Bubble':>10} {'Peak Mem':>10} {'Comm Factor':>12} {'Key Idea':>20}")
print(f"{'-'*80}")

data = [
    ("Naive", f"{(P-1)/P:.1%}", "1 MB", "1x", "Sequential"),
    ("GPipe", f"{(P-1)/(P-1+M):.1%}", f"{M} MB", "1x", "All-fwd then all-bwd"),
    ("1F1B", f"{(P-1)/(P-1+M):.1%}", f"{P} MB", "1x", "Interleave fwd/bwd"),
    ("Interleaved (v=2)", f"{(P-1)/(2*(P-1+M)):.1%}", f"{P} MB", "~2x", "Virtual stages"),
    ("Interleaved (v=4)", f"{(P-1)/(4*(P-1+M)):.1%}", f"{P} MB", "~4x", "More virtual stages"),
]

for name, bubble, mem, comm, idea in data:
    print(f"{name:<25} {bubble:>10} {mem:>10} {comm:>12} {idea:>20}")

In [ ]:
#@title 🎧 Listen: 3d Parallelism Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_3d_parallelism_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. 3D Parallelism: Putting It All Together

In practice, large models use a combination of all three parallelism strategies:
- **Tensor Parallelism** within a node (fast NVLink)
- **Pipeline Parallelism** across nodes (moderate bandwidth)
- **Data Parallelism** across pipeline groups (allreduce)

Let us calculate the total GPU count and communication for a realistic setup.

In [ ]:
#@title 🎧 Code Walkthrough: 3d Parallelism Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_3d_parallelism_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def analyze_3d_parallelism(total_gpus: int, tp: int, pp: int,
                            model_layers: int, hidden_dim: int,
                            seq_len: int = 4096, micro_batch: int = 1,
                            num_microbatches: int = 32):
    """Analyze a 3D parallelism configuration."""
    dp = total_gpus // (tp * pp)

    layers_per_stage = model_layers // pp
    global_batch = dp * num_microbatches * micro_batch

    # Pipeline bubble
    bubble = (pp - 1) / (pp - 1 + num_microbatches)

    # Activation size per pipeline transfer
    act_mb = micro_batch * seq_len * (hidden_dim // tp) * 2 / (1024**2)  # FP16, after TP split

    print(f"3D Parallelism Configuration")
    print(f"{'='*50}")
    print(f"  Total GPUs:        {total_gpus}")
    print(f"  Tensor Parallel:   {tp} (within node)")
    print(f"  Pipeline Parallel: {pp} (across nodes)")
    print(f"  Data Parallel:     {dp} (outer level)")
    print(f"")
    print(f"  Model layers:      {model_layers}")
    print(f"  Layers per stage:  {layers_per_stage}")
    print(f"  Hidden dim:        {hidden_dim} (effective: {hidden_dim // tp} per GPU)")
    print(f"")
    print(f"  Micro-batches:     {num_microbatches}")
    print(f"  Global batch:      {global_batch}")
    print(f"")
    print(f"  Pipeline bubble:   {bubble:.1%}")
    print(f"  Activation per transfer: {act_mb:.1f} MB")

# Example: LLaMA 70B on 64 GPUs
print("Example: LLaMA 70B on 64 GPUs")
print()
analyze_3d_parallelism(
    total_gpus=64, tp=8, pp=4,
    model_layers=80, hidden_dim=8192,
    num_microbatches=32
)

print("\n" + "="*50)
print("\nExample: GPT-3 175B on 1024 GPUs")
print()
analyze_3d_parallelism(
    total_gpus=1024, tp=8, pp=16,
    model_layers=96, hidden_dim=12288,
    num_microbatches=64
)

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_todo_1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Your Turn

### Exercise 1: Verify the v-factor Reduction

For $P = 4$ and $M = 16$, compute the bubble fraction for $v = 1, 2, 3, 4, 8$ and verify that each doubling of $v$ halves the bubble.

In [ ]:
#@title 🎧 Before You Start: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_todo_1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Compute bubble fractions for different v values
P = 4
M = 16

print(f"Bubble fractions for P={P}, M={M}:")
print(f"{'v':>4} {'Bubble':>10} {'Ratio to v=1':>14}")
print("-" * 32)

for v in [1, 2, 3, 4, 8]:
    # TODO: Compute bubble fraction
    bubble = None  # REPLACE

    # TODO: Compute ratio to v=1 case
    bubble_v1 = None  # REPLACE
    ratio = None  # REPLACE

    print(f"{v:>4} {bubble:>10} {ratio:>14}")

# QUESTION: Is v=8 always worth it? What factors limit how large v can be?

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_todo_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 2: Find the Optimal v

Given a model with $L = 96$ layers on $P = 8$ GPUs with $M = 32$ micro-batches, find the largest possible $v$ such that each virtual stage has at least 2 layers. Then compute the bubble fraction.

In [ ]:
#@title 🎧 Before You Start: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_todo_2_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Find the optimal v

L = 96    # total layers
P = 8     # GPUs
M = 32    # micro-batches
min_layers_per_vs = 2  # minimum layers per virtual stage

# layers_per_virtual_stage = L / (P * v)
# We need: L / (P * v) >= min_layers_per_vs
# So: v <= L / (P * min_layers_per_vs)

# TODO: Compute max_v
max_v = None  # REPLACE

# TODO: Compute bubble fractions for v=1 through max_v
# Which v gives the best bubble? Is it always max_v?

print(f"Model: {L} layers, {P} GPUs, {M} micro-batches")
print(f"Max v = {max_v}")
print(f"Layers per virtual stage at max v: {L / (P * max_v) if max_v else '???'}")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_todo_3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 3: Design a 3D Parallelism Configuration

You have 256 GPUs (32 nodes of 8 GPUs each) and need to train a 175B parameter model with 96 layers and hidden dimension 12288. Design a 3D parallelism configuration.

In [ ]:
#@title 🎧 Before You Start: Todo Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_todo_3_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Design your configuration
#
# Constraints:
# - TP should be <= 8 (within one node)
# - PP should divide the 96 layers evenly
# - DP = total_gpus / (TP * PP)
# - Choose M so that bubble < 10%
#
# Hint: Start with TP=8 (one full node), then choose PP

total_gpus = 256
model_layers = 96
hidden_dim = 12288

# TODO: Set your values
tp = None  # REPLACE
pp = None  # REPLACE
dp = None  # REPLACE: total_gpus // (tp * pp)
M = None   # REPLACE: choose M for < 10% bubble

# Verify: tp * pp * dp == total_gpus
# Verify: model_layers % pp == 0
# Verify: (pp - 1) / (pp - 1 + M) < 0.10

print(f"Your 3D config: TP={tp}, PP={pp}, DP={dp}")
print(f"Micro-batches: M={M}")
# analyze_3d_parallelism(total_gpus, tp, pp, model_layers, hidden_dim, num_microbatches=M)

In [ ]:
#@title 🎧 Listen: Summary Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_summary_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Summary

| Schedule | Bubble | Peak Memory | Communication | Best For |
|---|---|---|---|---|
| Naive | $(P-1)/P$ | 1 MB | 1x | Never use |
| GPipe | $(P-1)/(P-1+M)$ | $M$ per stage | 1x | Historical |
| 1F1B | $(P-1)/(P-1+M)$ | $P$ per stage | 1x | Standard choice |
| Interleaved ($v$) | $(P-1)/[v(P-1+M)]$ | $P$ per stage | $\sim v$x | Intra-node PP |

**Key takeaways:**

1. **Interleaved 1F1B** reduces the bubble by a factor of $v$ compared to standard 1F1B.
2. The cost is $v$ times more point-to-point communications.
3. This trade-off is worthwhile with **fast interconnects** (NVLink within a node).
4. In 3D parallelism setups, pipeline parallelism handles the **cross-node** dimension.
5. The progression from naive to interleaved 1F1B represents the complete story of how to efficiently split model depth across GPUs.

In [ ]:
#@title 🎧 Wrap-Up: Summary Conclusion
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_summary_conclusion.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("=" * 60)
print("SUMMARY: Pipeline Parallelism Complete")
print("=" * 60)
print()
print("The four schedules, in order of sophistication:")
print("  1. Naive:       Simple but terrible (75% bubble at P=4)")
print("  2. GPipe:       Micro-batching reduces bubble, but memory grows with M")
print("  3. 1F1B:        Same bubble, memory bounded at P -- the standard choice")
print("  4. Interleaved: Bubble reduced by v, at cost of more communication")
print()
print("Pipeline parallelism complements:")
print("  - Tensor parallelism (splits WIDTH within a node)")
print("  - Data parallelism (splits DATA across groups)")
print()
print("Together, these three form the backbone of every")
print("large-scale LLM training system in production today.")